In [2]:
import numpy as np
from scipy import linalg
from scipy.special import gamma

# from math import gamma, pi, log

In [ ]:
def group_lasso_density(vec, lambda_):
    size = len(vec)
    norm = linalg.norm(vec)
    density = (
        2 ** (-size)
        * np.pi ** (-(size - 1) / 2)
        / gamma((size + 1) / 2)
        * np.exp(-norm * lambda_)
    )
    return density


def p_star(vec, theta, lambda0, lambda1):
    spike = (1 - theta) * group_lasso_density(vec, lambda0)
    slab = theta * group_lasso_density(vec, lambda1)
    p_star_value = slab / (spike + slab)
    return p_star_value


def lambda_star(vec, theta, lambda0, lambda1):
    p_star_value = p_star(vec, theta, lambda0, lambda1)
    lambda_star_value = (1 - p_star_value) * lambda0 + p_star_value * lambda1
    return lambda_star_value


def update_momentum(x, x_lag, iter):
    momentum = x + (iter - 2) / (iter + 1) * (x - x_lag)
    return momentum


def h_function(lambda_star_value, p_star_value, lambda1, eta):
    h_value = (lambda_star_value - lambda1) ** 2 + 2 / eta * p_star_value
    return h_value


def update_delta(h_value, eta, p_star_value, lambda0, lambda1):
    if h_value > 0:
        delta = np.sqrt(2 * eta * np.log(1 / p_star_value)) + eta * lambda1
    else:
        delta = eta * lambda_star(p_star_value, lambda0, lambda1)
    return delta


def update_count(mat):
    count = 0
    d = mat.shape[1]
    for i in range(d):
        if linalg.norm(mat[:, i], 0) != 0:
            count += 1
    return count, d


def update_theta(count, d, alpha, beta):
    theta = (alpha + count) / (alpha + beta + d)
    return theta

In [ ]:
def get_W(Y, mu, U, V, A, B, xi):
    W = (1 + xi * Y + Y) / (
        1 + np.exp(-np.outer(mu, np.ones(Y.shape[1])) - U @ A @ B.T @ V.T)
    )
    return W


def gradient(side, Y, mu, U, V, A, B, xi):
    W = get_W(Y, mu, U, V, A, B, xi)
    if side == 'A':
        grad = B.T @ V.T @ (xi * Y - W).T @ U
    elif side == 'B':
        grad = A.T @ U.T @ (xi * Y - W) @ V
    return grad


def log_likelihood(Y, mu, U, V, A, B, xi, theta, lambda0, lambda1, tilde_theta, tilde_lambda0, tilde_lambda1):
    M = np.outer(mu, np.ones(Y.shape[1])) + U @ A @ B.T @ V.T
    d1 = U.shape[1]
    d2 = V.shape[1]
    LambdaStarA = np.diag([lambda_star(A[i, :], tilde_theta, tilde_lambda0, tilde_lambda1) for i in range(d1)])
    LambdaStarB = np.diag([lambda_star(B[j, :], theta, lambda0, lambda1) for j in range(d2)])
    penalty = np.sum(np.dot(LambdaStarA, A)) + np.sum(np.dot(LambdaStarB, B))

    loglik = np.sum(xi*Y * M - (1+xi*Y-Y)*np.log(1 + np.exp(M))) - penalty
    return loglik

def optimization(Y, mu, U, V, A = None, B = None, K, xi, eta, seed = None, max_iter=100, tol=1e-3):

    d1 = U.shape[1]
    d2 = V.shape[1]

    if K is None:
        


    if A is None:
        if seed is not None:
            np.random.seed(seed)
            A = np.random.normal(0, 1, (K, d1))
        d1 = U.shape[1]
    np.random.normal(0, 1, A.shape)
    A_lag = A.copy()
    B_lag = B.copy()
    for iter in range(1, max_iter + 1):
        # update A
        A_momentum = update_momentum(A, A_lag, iter)
        grad_A = gradient('A', Y, mu, U, V, A_momentum, B, xi)
        A_new = A_momentum + eta * grad_A
        A_lag = A.copy()
        A = A_new.copy()

        # update B
        B_momentum = update_momentum(B, B, iter)
        grad_B = gradient('B', Y, mu, U, V, A, B_momentum, xi)
        B_new = B_momentum + eta * grad_B
        B_lag = B.copy()
        B = B_new.copy()

        # check convergence
        norm_A = linalg.norm(A - A_lag) / (linalg.norm(A_lag) + 1e-8)
        norm_B = linalg.norm(B - B_lag) / (linalg.norm(B_lag) + 1e-8)
        if max(norm_A, norm_B) < tol:
            break

    return A, B




np.float64(2.302585092994046)

In [5]:
u = np.diag(np.array([1, 2, 3]))
v = np.array([[1, 1, 1], [5, 5, 5], [6, 6, 6]])
rank1_matrix = np.dot(u, v)
rank1_matrix

array([[ 1,  1,  1],
       [10, 10, 10],
       [18, 18, 18]])

In [ ]:
# define matrix completion problem
import numpy as np
from scipy.linalg import svd


def matrix_completion(M, mask, rank, n_iter=100, tol=1e-4):
    """
    Perform matrix completion using singular value thresholding.
    """
    # Initialize the completed matrix
    X = np.zeros_like(M)
    for i in range(n_iter):
        # Perform SVD
        U, s, Vt = svd(X, full_matrices=False)
        # Keep only the largest singular values
        s = np.maximum(s - tol, 0)
        # Reconstruct the matrix
        X = np.dot(U, np.dot(np.diag(s), Vt))
        # Apply the mask
        X *= mask
        # Check for convergence
        if np.linalg.norm(M - X) < tol:
            break
    return X